# Assignment 9.1 — Tabular Q-Learning

You will train a **tabular Q-learning** agent on `arena.JumperEnv`.
The action policy is ε-greedy; ε decays over training. The Q-update
is the off-policy TD update of Watkins (1989):

$$
Q(s, a) \leftarrow Q(s, a) + \alpha \cdot \big[\, r + \gamma \cdot \max_{a'} Q(s', a') - Q(s, a) \,\big]
$$

The catch: the env returns a **continuous** 5-dimensional observation
`(player_height, player_vy, next_obs_dx, next_obs_w, speed)`. A
Q-table needs a **finite, discrete** state space, so the first job
is a sensible discretization (binning).

## Tasks

Fill in the `TODO` blocks in the cells below:

1. **`discretize(obs)`** — map the 5-dim float observation to a
   tuple of bin indices. Suggested bin counts are given as
   defaults; you may change them.
2. **`Agent.act(state, greedy)`** — ε-greedy action selection.
3. **`Agent.update(s, a, r, s_next, done)`** — apply the Q-learning
   update above. When `done=True`, the bootstrap term
   $\max_{a'} Q(s', a')$ is **0**.
4. **`train(...)`** — the outer loop. For each episode: reset, roll
   out until `terminated or truncated`, call `update` after every
   step, decay ε, log the return.

## Hints

- Defaults: ε starts at 1.0, decays multiplicatively to 0.02; α=0.2;
  γ=0.99; ~5000 episodes. Expect avg-100 return to stay near −1 for
  the first ~1500 episodes (pure exploration), then climb sharply.
- `next_obs_dx` is by far the most important feature. Bin it finely
  close to the player (where jump timing matters) and coarsely far
  away.
- A converged agent should reach ≥ 1000 steps and score ≥ 40 in
  greedy eval.

In [ ]:
from __future__ import annotations
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Callable

import numpy as np
import matplotlib.pyplot as plt

from arena import JumperEnv

## 1. Discretization

The first/last bins are open-ended (`(-inf, edge0]` and
`(edgeN, +inf)`); `np.digitize` handles that automatically. `DX` is
binned finely near the player (where jump timing matters) — the
jump-trigger window for this env is ~`dx ∈ (8, 16]`.

In [ ]:
HEIGHT_EDGES = np.array([1.0, 6.0, 14.0, 22.0])                          # 5 bins
VY_EDGES     = np.array([-4.0, -1.0, 0.5, 2.0, 4.0])                     # 6 bins
DX_EDGES     = np.array([4.0, 8.0, 12.0, 18.0, 26.0, 38.0, 55.0, 80.0])  # 9 bins
WIDTH_EDGES  = np.array([5.5, 7.5])                                      # 3 bins
SPEED_EDGES  = np.array([2.5, 3.5, 4.5])                                 # 4 bins

def discretize(obs: np.ndarray) -> tuple[int, int, int, int, int]:
    """
    Map a 5-dim continuous observation to a tuple of bin indices.

    obs layout: (player_height, player_vy, next_obs_dx, next_obs_w, speed)

    TODO: return a tuple of five ints, one per feature, using
    np.digitize and the edge arrays above.
    """
    raise NotImplementedError("discretize")

## 2. The agent

In [ ]:
@dataclass
class Agent:
    n_actions: int = 2
    alpha: float = 0.2
    gamma: float = 0.99
    epsilon: float = 1.0
    epsilon_min: float = 0.02
    epsilon_decay: float = 0.9985    # multiplicative, applied per episode
    seed: int | None = None

    Q: dict[tuple, np.ndarray] = field(
        default_factory=lambda: defaultdict(lambda: np.zeros(2)))
    rng: np.random.Generator = field(init=False)

    def __post_init__(self) -> None:
        self.rng = np.random.default_rng(self.seed)

    def act(self, state: tuple, greedy: bool = False) -> int:
        """
        ε-greedy action selection. If greedy=True, ignore ε and always
        pick argmax_a Q(state, a).

        TODO:
          - if not greedy and rng.random() < epsilon: return a uniform random action
          - otherwise return int(np.argmax(self.Q[state]))
        """
        raise NotImplementedError("Agent.act")

    def update(self, s: tuple, a: int, r: float, s_next: tuple, done: bool) -> None:
        """
        Apply the Q-learning update:

            target = r                                  if done
                     r + gamma * max_a' Q(s_next, a')   otherwise
            Q[s][a] += alpha * (target - Q[s][a])

        TODO: implement.
        """
        raise NotImplementedError("Agent.update")

    def decay_epsilon(self) -> None:
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

## 3. Training loop

In [ ]:
def train(env: JumperEnv,
          agent: Agent,
          n_episodes: int = 5000,
          on_episode: Callable[[int, float, int], None] | None = None,
          ) -> list[float]:
    """
    Run `n_episodes` of training. After every episode, call
    on_episode(ep, ret, steps). Returns the list of episode returns.

    TODO:
      for each episode:
          obs, _ = env.reset()
          state = discretize(obs)
          ep_return, steps = 0.0, 0
          while True:
              a = agent.act(state)
              next_obs, r, term, trunc, _ = env.step(a)
              next_state = discretize(next_obs)
              agent.update(state, a, r, next_state, term)
              ep_return += r
              steps += 1
              state = next_state
              if term or trunc:
                  break
          agent.decay_epsilon()
          if on_episode is not None:
              on_episode(ep, ep_return, steps)
    """
    raise NotImplementedError("train")

## 4. Run training

Once your TODOs are filled in, run the cell below. On a laptop CPU
this should take well under a minute for 5000 episodes.

In [ ]:
env = JumperEnv(seed=0)
agent = Agent(seed=0)

returns: list[float] = []
def on_ep(ep: int, ret: float, steps: int) -> None:
    returns.append(ret)
    if (ep + 1) % 500 == 0:
        avg = np.mean(returns[-100:])
        print(f"ep {ep+1:5d}  return(avg100) {avg:7.2f}  steps {steps:4d}  "
              f"ε {agent.epsilon:.3f}  |Q|={len(agent.Q)}")

train(env, agent, n_episodes=5000, on_episode=on_ep)
print(f"\ndone. Q-table has {len(agent.Q)} states.")

## 5. Learning curve

In [ ]:
r = np.asarray(returns, dtype=np.float32)
window = 50
smoothed = np.convolve(r, np.ones(window) / window, mode="valid")
plt.figure(figsize=(8, 4))
plt.plot(r, alpha=0.25, label="episode return")
plt.plot(np.arange(len(smoothed)) + window - 1, smoothed,
         label=f"rolling mean (w={window})")
plt.xlabel("episode"); plt.ylabel("return")
plt.title("Q-learning on JumperEnv")
plt.legend(); plt.tight_layout(); plt.show()

## 6. Greedy evaluation

Disable exploration and roll out 10 episodes from different seeds.
A converged agent should average **≥ 1000 steps** and score **≥ 40**.

In [ ]:
steps_log, score_log = [], []
for trial in range(10):
    obs, _ = env.reset(seed=100 + trial)
    state = discretize(obs)
    steps, ret = 0, 0.0
    while True:
        a = agent.act(state, greedy=True)
        obs, r, term, trunc, info = env.step(a)
        state = discretize(obs)
        ret += r; steps += 1
        if term or trunc:
            break
    steps_log.append(steps); score_log.append(info["score"])
    print(f"  trial {trial}: steps={steps:5d}  return={ret:7.2f}  score={info['score']}")

print(f"\nmean steps {np.mean(steps_log):.1f}   mean score {np.mean(score_log):.1f}")

## 7. Watch a learned episode

Roll out one greedy episode and play it back as an embedded
animation. The cap is 250 frames (~8 s at 30 fps) so the notebook
stays small even when the agent runs forever. Watch how late your
agent commits to a jump as the world speed grows.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

play_env = JumperEnv(seed=200)
obs, _ = play_env.reset()
state = discretize(obs)
frames = [play_env.render(mode="rgb_array")]
actions: list[int] = []
last_info = {"score": 0}
while len(frames) < 250:
    a = agent.act(state, greedy=True)
    actions.append(a)
    obs, r, term, trunc, last_info = play_env.step(a)
    state = discretize(obs)
    frames.append(play_env.render(mode="rgb_array"))
    if term or trunc:
        break
play_env.close()

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.axis("off")
im = ax.imshow(frames[0])

def _update(i):
    im.set_array(frames[i])
    return [im]

anim = animation.FuncAnimation(
    fig, _update, frames=len(frames), interval=33, blit=True)
plt.close(fig)
print(f"frames {len(frames)}   score {last_info['score']}   "
      f"jumps {sum(actions)}/{len(actions)}")
HTML(anim.to_jshtml())

## 8. Reflection

Answer briefly (in a markdown cell or a short `NOTES.md`):

1. Which bins did you choose for `next_obs_dx` and `player_vy`, and
   why? What happens to learning if you make the `next_obs_dx` bins
   too coarse?
2. What is the largest `α` (learning rate) that still converges?
   What happens above it?
3. Roughly how many episodes does your agent need to consistently
   survive ≥ 500 steps? How does this compare to the random baseline
   (~70 steps)?

## Save the table for later

Optional — pickle the Q-table so you can reload it without retraining.

In [ ]:
import pickle
from pathlib import Path

out = Path("q_table.pkl")
with out.open("wb") as f:
    pickle.dump(dict(agent.Q), f)
print(f"saved {out} ({len(agent.Q)} states)")

Continue to [`2-deep_q_learning.ipynb`](2-deep_q_learning.ipynb) for
the DQN extension.